In [ ]:
import json
import requests
import os

from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import display, HTML

load_dotenv()
client = OpenAI(
    base_url="https://api.deepseek.com", api_key=os.getenv("DEEPSEEK_API_KEY")
)

In [2]:
import xml.etree.ElementTree as ET
from tavily import TavilyClient

load_dotenv()

session = requests.Session()
session.headers.update({
    "User-Agent": "LF-ADP-Agent/1.0 (mailto:your.email@example.com)"
})


def arxiv_search_tool(query: str, max_results: int = 5) -> list[dict]:
    """
    Searches arXiv for research papers matching the given query.
    """
    url = f"https://export.arxiv.org/api/query?search_query=all:{query}&start=0&max_results={max_results}"

    try:
        response = session.get(url, timeout=30)
        response.raise_for_status()
    except requests.exceptions.RequestException as e:
        return [{"error": str(e)}]

    try:
        root = ET.fromstring(response.content)
        ns = {'atom': 'http://www.w3.org/2005/Atom'}

        results = []
        for entry in root.findall('atom:entry', ns):
            title = entry.find('atom:title', ns).text.strip()
            authors = [author.find('atom:name', ns).text for author in entry.findall('atom:author', ns)]
            published = entry.find('atom:published', ns).text[:10]
            url_abstract = entry.find('atom:id', ns).text
            summary = entry.find('atom:summary', ns).text.strip()

            link_pdf = None
            for link in entry.findall('atom:link', ns):
                if link.attrib.get('title') == 'pdf':
                    link_pdf = link.attrib.get('href')
                    break

            results.append({
                "title": title,
                "authors": authors,
                "published": published,
                "url": url_abstract,
                "summary": summary,
                "link_pdf": link_pdf
            })

        return results
    except Exception as e:
        return [{"error": f"Parsing failed: {str(e)}"}]
    

arxiv_tool_def = {
    "type": "function",
    "function": {
        "name": "arxiv_search_tool",
        "description": "Searches for research papers on arXiv by query string.",
        "parameters": {
            "type": "object",
            "properties": {
                "query": {
                    "type": "string",
                    "description": "Search keywords for research papers."
                },
                "max_results": {
                    "type": "integer",
                    "description": "Maximum number of results to return.",
                    "default": 5
                }
            },
            "required": ["query"]
        }
    }
}

In [3]:
# Test the arXiv search tool
results = arxiv_search_tool("retrieval-augmented generation", max_results=3)

# Show formatted results
for i, paper in enumerate(results, 1):
    if "error" in paper:
        print(f"❌ Error: {paper['error']}")
    else:
        print(f"📄 Paper {i}")
        print(f"  Title     : {paper['title']}")
        print(f"  Authors   : {', '.join(paper['authors'])}")
        print(f"  Published : {paper['published']}")
        print(f"  URL       : {paper['url']}\n")

#print("\n🧾 Raw Results:\n")
#print(json.dumps(results, indent=2))

📄 Paper 1
  Title     : AR-RAG: Autoregressive Retrieval Augmentation for Image Generation
  Authors   : Jingyuan Qi, Zhiyang Xu, Qifan Wang, Lifu Huang
  Published : 2025-06-08
  URL       : http://arxiv.org/abs/2506.06962v3

📄 Paper 2
  Title     : EVOR: Evolving Retrieval for Code Generation
  Authors   : Hongjin Su, Shuyang Jiang, Yuhang Lai, Haoyuan Wu, Boao Shi, Che Liu, Qian Liu, Tao Yu
  Published : 2024-02-19
  URL       : http://arxiv.org/abs/2402.12317v2

📄 Paper 3
  Title     : Automated Literature Review Using NLP Techniques and LLM-Based Retrieval-Augmented Generation
  Authors   : Nurshat Fateh Ali, Md. Mahdi Mohtasim, Shakil Mosharrof, T. Gopi Krishna
  Published : 2024-11-27
  URL       : http://arxiv.org/abs/2411.18583v1



In [ ]:
def tavily_search_tool(query: str, max_results: int = 5, include_images: bool = False) -> list[dict]:
    """
    Perform a search using the Tavily API.

    Args:
        query (str): The search query.
        max_results (int): Number of results to return (default 5).
        include_images (bool): Whether to include image results.

    Returns:
        list[dict]: A list of dictionaries with keys like 'title', 'content', and 'url'.
    """
    api_key = os.getenv("TAVILY_API_KEY")
    if not api_key:
        raise ValueError("TAVILY_API_KEY not found in environment variables.")

    client = TavilyClient(api_key=api_key)

    try:
        response = client.search(
            query=query,
            max_results=max_results,
            include_images=include_images
        )

        results = []
        for r in response.get("results", []):
            results.append({
                "title": r.get("title", ""),
                "content": r.get("content", ""),
                "url": r.get("url", "")
            })

        if include_images:
            for img_url in response.get("images", []):
                results.append({"image_url": img_url})

        return results

    except Exception as e:
        return [{"error": str(e)}]  # For LLM-friendly agents


tavily_tool_def = {
    "type": "function",
    "function": {
        "name": "tavily_search_tool",
        "description": "Performs a general-purpose web search using the Tavily API.",
        "parameters": {
            "type": "object",
            "properties": {
                "query": {
                    "type": "string",
                    "description": "Search keywords for retrieving information from the web."
                },
                "max_results": {
                    "type": "integer",
                    "description": "Maximum number of results to return.",
                    "default": 5
                },
                "include_images": {
                    "type": "boolean",
                    "description": "Whether to include image results.",
                    "default": False
                }
            },
            "required": ["query"]
        }
    }
}

In [5]:
# Test the Tavily search tool
search_results = tavily_search_tool("retrieval-augmented generation applications")
for item in search_results:
    print(f"  Title     : {item['title']}")
    print(f"  Content   : {item['content'][:80]} [truncated]")
    print(f"  URL       : {item['url']}\n")

  Title     : Retrieval-augmented generation - Wikipedia
  Content   : *   [(Top)](https://en.wikipedia.org/wiki/Retrieval-augmented_generation#). *    [truncated]
  URL       : https://en.wikipedia.org/wiki/Retrieval-augmented_generation

  Title     : What Is Retrieval-Augmented Generation aka RAG - NVIDIA Blog
  Content   : ![NVIDIA artist's concept of retrieval-augmented generation aka RAG](https://blo [truncated]
  URL       : https://blogs.nvidia.com/blog/what-is-retrieval-augmented-generation

  Title     : What is Retrieval-Augmented Generation (RAG)? - Google Cloud
  Content   : *   [Topics](https://cloud.google.com/discover/). [Find a partner](https://cloud [truncated]
  URL       : https://cloud.google.com/use-cases/retrieval-augmented-generation

  Title     : Retrieval-Augmented Generation (RAG) - Pinecone
  Content   : In this article, we’ll explore the limitations of foundation models and how retr [truncated]
  URL       : https://www.pinecone.io/learn/retrieval-augmente

In [6]:
tool_mapping = {
    "tavily_search_tool": tavily_search_tool,
    "arxiv_search_tool": arxiv_search_tool,
}

In [7]:
def generate_research_report_with_tools(prompt_: str, model: str = "deepseek-v4-flash") -> str:
    """
    Generates a research report using OpenAI's tool-calling with arXiv and Tavily tools.

    Args:
        prompt_ (str): The user prompt.
        model (str): OpenAI model name.

    Returns:
        str: Final assistant research report text.
    """
    messages = [
        {
            "role": "system",
            "content": (
                "You are a research assistant that can search the web and arXiv to write detailed, "
                "accurate, and properly sourced research reports.\n\n"
                "🔍 Use tools when appropriate (e.g., to find scientific papers or web content).\n"
                "📚 Cite sources whenever relevant. Do NOT omit citations for brevity.\n"
                "🌐 When possible, include full URLs (arXiv links, web sources, etc.).\n"
                "✍️ Use an academic tone, organize output into clearly labeled sections, and include "
                "inline citations or footnotes as needed.\n"
                "🚫 Do not include placeholder text such as '(citation needed)' or '(citations omitted)'."
            )
        },
        {"role": "user", "content": prompt_}
    ]

    functions = [arxiv_tool_def, tavily_tool_def]
    MAX_TURNS = 10
    final_text = None
    
    for _ in range(MAX_TURNS):
        response = client.chat.completions.create(
            model=model,
            messages=messages,
            tools=functions,
            tool_choice="auto",
            temperature=1,
        )

        msg = response.choices[0].message
        messages.append(msg)

        if not msg.tool_calls:
            final_text = msg.content
            print("✅ Final answer:")
            print(final_text)
            break

        for call in msg.tool_calls:
            tool_name = call.function.name
            args = json.loads(call.function.arguments)
            print(f"🛠️ {tool_name}({args})")

            try:
                tool_func = tool_mapping[tool_name]
                result = tool_func(**args)
            except Exception as e:
                result = {"error": str(e)}

            messages.append({
                "role": "tool",
                "tool_call_id": call.id,
                "name": tool_name,
                "content": json.dumps(result)
            })

    return final_text or ""

In [8]:
def reflection_and_rewrite(text_or_messages, model: str = "deepseek-v4-flash", temperature: float = 0.3) -> dict:
    """
    Generates a structured reflection AND a revised research report.
    Accepts raw text OR the messages list returned by generate_research_report_with_tools.

    Returns:
        dict with keys:
          - "reflection": structured reflection text
          - "revised_report": improved version of the input report
    """
    # Extract assistant content if messages were passed
    if isinstance(text_or_messages, list):
        text = None
        for m in reversed(text_or_messages):
            role = m.get("role") if isinstance(m, dict) else getattr(m, "role", None)
            content = m.get("content") if isinstance(m, dict) else getattr(m, "content", None)
            if role == "assistant" and content:
                text = content
                break
        if not text:
            raise ValueError("No assistant text found in messages.")
    else:
        text = str(text_or_messages)

    # Ask for both reflection + rewritten report
    user_prompt = (
        "First, provide a structured reflection (Strengths, Limitations, Suggestions, Opportunities) "
        "on the following report.\n\n"
        "Then, write a revised version of the report that incorporates your suggestions, "
        "improves clarity, and strengthens academic tone.\n\n"
        f"Report:\n{text}"
    )

    resp = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": "You are an academic reviewer and editor."},
            {"role": "user", "content": user_prompt},
        ],
        temperature=temperature,
    )

    # Expect the model to produce both sections in one response
    full_output = resp.choices[0].message.content.strip()

    return {
        "reflection": full_output,   # includes reflection
        "revised_report": full_output  # same output may include both parts
    }

In [9]:
def convert_report_to_html(text_or_messages, model: str = "deepseek-v4-flash") -> str:
    """
    Converts a plaintext research report into a styled HTML page using OpenAI.
    Accepts raw text OR the messages list from the tool-calling step.
    """
    # Inline extraction (sin helper)
    if isinstance(text_or_messages, list):
        text_report = None
        for m in reversed(text_or_messages):
            role = m.get("role") if isinstance(m, dict) else getattr(m, "role", None)
            content = m.get("content") if isinstance(m, dict) else getattr(m, "content", None)
            if role == "assistant" and content:
                text_report = content
                break
        if not text_report:
            raise ValueError("No assistant text found in messages.")
    else:
        text_report = str(text_or_messages)

    if not text_report:
        raise ValueError("Empty report text.")

    system_prompt = "You convert plaintext reports into full clean HTML documents."
    user_prompt = (
        "You are an expert technical writing assistant. "
        "Convert the following plaintext research report into a clean, structured HTML document. "
        "Include section headers, well-formatted paragraphs, inline links, and a clean readable layout. "
        "Ensure that all URLs are clickable and citation style is preserved.\n\n"
        "Respond ONLY with a valid HTML snippet, starting with '```html' and ending with '```'.\n\n"
        f"Report:\n{text_report}"
    )

    resp = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        temperature=0.5,
    )
    return resp.choices[0].message.content.strip()

In [10]:
# 1) Research with tools

In [11]:
prompt_ = "Radio observations of recurrent novae"
preliminary_report = generate_research_report_with_tools(prompt_)
print(preliminary_report)

🛠️ arxiv_search_tool({'query': 'radio observations recurrent novae', 'max_results': 10})
🛠️ tavily_search_tool({'query': 'radio observations of recurrent novae', 'max_results': 10})
🛠️ arxiv_search_tool({'query': 'RS Ophiuchi radio observations recurrent nova', 'max_results': 10})
🛠️ tavily_search_tool({'query': 'radio emission recurrent novae RS Oph V3890 Sgr synchrotron thermal', 'max_results': 10})
🛠️ arxiv_search_tool({'query': 'T Pyxidis radio observations recurrent nova', 'max_results': 5})
🛠️ tavily_search_tool({'query': 'classical novae radio observations VLA e-Nova survey synchrotron', 'max_results': 5})
✅ Final answer:
# Radio Observations of Recurrent Novae

## 1. Introduction

Recurrent novae (RNe) are a subclass of cataclysmic variable stars in which a white dwarf (WD) accretes matter from a companion star and undergoes repeated thermonuclear runaway (TNR) explosions on its surface. Unlike classical novae, which have recurrence timescales of \( \sim 10^4 \)–\( 10^5 \) year

In [12]:
# 2) Reflection on the report (use the final TEXT to avoid ambiguity)
reflection_text = reflection_and_rewrite(preliminary_report)
print(reflection_text['reflection'])

## Structured Reflection

**Strengths:**
- Comprehensive coverage of key recurrent novae (RS Oph, V745 Sco, V3890 Sgr, T Pyx) with detailed observational results
- Clear differentiation of thermal and non-thermal emission mechanisms, supported by specific brightness temperature values
- Effective use of summary table to consolidate key physical properties
- Good integration of multi-wavelength context (X-ray, gamma-ray) and facility descriptions
- Citations to primary literature with accessible links

**Limitations:**
- Citation format (inline hyperlinks) is non-standard for academic writing; lacks author–year references and bibliography
- Introduction does not fully motivate why *radio* observations are uniquely important for recurrent novae relative to classical novae or other wavelengths
- Some passages are verbose or contain informal phrasing (e.g., "decisively ruling out", "hinting at a common site")
- Section ordering could be improved: emission mechanisms are described before th

In [13]:
# 3) Convert the report to HTML (use the TEXT and correct function name)
html = convert_report_to_html(reflection_text['revised_report'])

In [14]:
print((html or "")[:200], "\n... [truncated]\n")

```html
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Radio Observations of Recurrent Novae</ 
... [truncated]



In [15]:
print("[truncated] ...\n", (html or "")[-200:])

[truncated] ...
 966.2007.12431.x</a></p>
    <p>Weston, J. H. S., et al. 2013, <em>MNRAS</em>, 434, 3390 — <a href="https://doi.org/10.1093/mnras/stt1829">doi:10.1093/mnras/stt1829</a></p>
</div>

</body>
</html>
```


In [17]:
# 4) Display full HTML
display(HTML(html[7: -3]))

Property,Finding
Emission mechanism,Synchrotron-dominated in symbiotic RNe; thermal + synchrotron in others
Ejecta mass,\( 10^{-5}\)–\(10^{-6} M_{\odot} \) (RNe); often lower than classical novae
Morphology,Bipolar/jet-like; asymmetric; collimated by binary
Brightness temperature,\( T_B \sim 10^{7} \) K (synchrotron) vs. \( \sim 10^{4} \) K (thermal)
Recurrence effects,Decreasing foreground density with successive outbursts
Super-remnants,\( \sim 10\)–100 pc structures built up over millennia
